# SDG Hub — Seed Data Semi-automático (Notebook de estudos)

Este notebook cria um **pequeno** `seed_data.jsonl` no formato esperado pelo **SDG Hub** e, em seguida, roda um flow de geração de dados a partir desse seed.

> Objetivo: manter o **padrão do SDG Hub** (seed JSONL → `FlowRegistry` → `Flow.from_yaml()` → `flow.generate()`), mas com **seed semi-automático**:  
> - `document_outline` gerado por LLM  
> - `icl_document` selecionado automaticamente (heurísticas simples)  
> - `icl_query_1..3` geradas por LLM e parseadas em colunas  
>
> Referências (padrão base):
> - `document_pre_processing.ipynb` (seed manual)  
> - `knowledge_generation.ipynb` (consumo do seed e execução de flows)

---

## Pré-requisitos

1. SDG Hub instalado (recomendado): `pip install sdg-hub[examples]`
2. `.env` com configuração do provider/modelo (ou variáveis de ambiente equivalentes)



In [ ]:
# Step 0: Instalação (se necessário)
# Observação: em ambientes já preparados, você pode pular esta célula.

%pip install -U sdg-hub[examples] python-dotenv datasets markdown-it-py nest_asyncio


In [1]:
# Step 0.1: Imports e .env
from dotenv import load_dotenv
import os
from pathlib import Path

load_dotenv()


True

In [2]:
# Required to run the flow with async mode (padrão do SDG Hub)
import nest_asyncio
nest_asyncio.apply()


## Step 1: Document Processing Pipeline (parse -> markdown)

Este passo replica o notebook oficial: usa o parser do InstructLab (Docling v2) para transformar arquivos em `.md`.

- Coloque documentos (PDF/DOCX/etc.) em `document_collection/`
- O comando abaixo gera `.md` na mesma pasta

> Se você **já** tem `.md`, pode pular e ir para o Step 2.


In [3]:
# Step 1: Document Processing Pipeline
data_dir = "document_collection/"

# Parser do exemplo oficial do sdg_hub (instructlab/docling)
# Se o arquivo ../docparser_v2.py não existir no seu checkout, ajuste o path conforme seu repo.
!python ../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../docling_v2_config.yaml


[14:10:25] INFO     Found 1 PDF files to process             ]8;id=658668;file:///opt/app-root/src/sdg_hub/examples/knowledge_tuning/enhanced_summary_knowledge_tuning/../docparser_v2.py\docparser_v2.py]8;;\:]8;id=511143;file:///opt/app-root/src/sdg_hub/examples/knowledge_tuning/enhanced_summary_knowledge_tuning/../docparser_v2.py#193\193]8;;\
           INFO     Processing document_collection/Feriados  ]8;id=525830;file:///opt/app-root/src/sdg_hub/examples/knowledge_tuning/enhanced_summary_knowledge_tuning/../docparser_v2.py\docparser_v2.py]8;;\:]8;id=578035;file:///opt/app-root/src/sdg_hub/examples/knowledge_tuning/enhanced_summary_knowledge_tuning/../docparser_v2.py#207\207]8;;\
                    e Emendas de 2026.pdf                                       
[INFO] 2026-02-27 14:10:25,785 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-27 14:10:25,788 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-02-27 14:10:25,798 [RapidOCR] d

## Step 2: Carregar um documento Markdown processado

Para estudo, vamos trabalhar com **um** `.md` (o primeiro encontrado).


In [4]:
import glob

md_files = sorted(glob.glob(f"{data_dir}/*.md"))
if not md_files:
    raise FileNotFoundError(f"Nenhum .md encontrado em {data_dir}. Rode o Step 1 ou coloque um .md lá.")

md_path = md_files[0]
print("Using:", md_path)

with open(md_path, "r", encoding="utf-8") as f:
    text = f.read()

print("Chars:", len(text))
print(text[:800])


Using: document_collection/Feriados e Emendas de 2026.md
Chars: 2274
## 1. Ficha técnica / Metadados

| Nome             | Feriados e Emendas de 2026         |
|------------------|------------------------------------|
| Área responsável | Departamento pessoal               |
| Categoria        | Política de governança corporativa |

## 1.1. Revisões

|   Versão | Data       | Alteração   |
|----------|------------|-------------|
|        5 | 03/02/2026 | Testes...   |
|        1 | 02/02/2026 | Teste 01    |

## 2. Objetivo

Divulgação do calendário de feriados nacionais e respectivos dias de emenda para o ano de 2026 .

## 3. Feriados

Informamos que o escritório estará fechado nas seguintes datas, que correspondem a feriados nacionais e dias de emenda (em amarelo):

| Periodo           | Data                  | Dia da Semana             | Feriado/Emenda   


## Step 3: Chunking (padrão compatível com o notebook do SDG Hub)

Vamos reutilizar a mesma ideia do exemplo: chunking em nível de blocos markdown e overlap.


In [5]:
from markdown_it import MarkdownIt
from typing import List
import re

def chunk_markdown(text: str, max_tokens: int = 800, overlap: int = 120) -> List[str]:
    """Chunking simples em nível de blocos (headings/parágrafos/listas etc.) + overlap em palavras."""
    md = MarkdownIt()
    tokens = md.parse(text)

    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
            buf = []
        elif tok.content:
            buf.append(tok.content)

    if buf:
        blocks.append("\n".join(buf).strip())

    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                chunks.append(" ".join(current_words))
                current_words = current_words[-overlap:] if overlap > 0 else []
    if current_words:
        chunks.append(" ".join(current_words))
    return [c.strip() for c in chunks if c.strip()]

chunks = chunk_markdown(text, max_tokens=int(os.getenv("CHUNK_MAX_TOKENS", "800")), overlap=int(os.getenv("CHUNK_OVERLAP", "120")))
print("chunks:", len(chunks))
print(chunks[0][:600])


chunks: 1
1. Ficha técnica / Metadados | Nome | Feriados e Emendas de 2026 | |------------------|------------------------------------| | Área responsável | Departamento pessoal | | Categoria | Política de governança corporativa | 1.1. Revisões | Versão | Data | Alteração | |----------|------------|-------------| | 5 | 03/02/2026 | Testes... | | 1 | 02/02/2026 | Teste 01 | 2. Objetivo Divulgação do calendário de feriados nacionais e respectivos dias de emenda para o ano de 2026 . 3. Feriados Informamos que o escritório estará fechado nas seguintes datas, que correspondem a feriados nacionais e dias de em


## Step 4: Seed semi-automático (outline + icl_document + icl_queries)

No seed “manual” do exemplo, você preenche na mão:
- `document_outline`
- `icl_document`
- `icl_query_1..3`
- `domain`

Aqui vamos:
1) Selecionar automaticamente um `icl_document` (heurísticas simples)
2) Gerar `document_outline` com LLM (via bloco SDG Hub)
3) Gerar 3 queries com LLM (via bloco SDG Hub) e parsear para colunas

### 4.1 Seleção automática do `icl_document`
Heurística (boa o bastante para começar pequeno):
- descartar chunks muito curtos
- penalizar boilerplate típico (copyright, “table of contents”, etc.)
- escolher o chunk com melhor score



In [6]:
def score_chunk(c: str) -> float:
    # Penaliza boilerplate comum
    boiler = [
        "table of contents", "copyright", "all rights reserved",
        "disclaimer", "confidential", "license", "spdx"
    ]
    c_low = c.lower()
    penalty = sum(1 for b in boiler if b in c_low)

    # Recompensa tamanho “útil”
    n = len(c.split())
    if n < 120:
        return -999

    # Penaliza excesso de linhas muito curtas (toc/lista de links)
    short_lines = sum(1 for line in c.splitlines() if 0 < len(line.strip()) < 20)
    return n - (penalty * 200) - (short_lines * 5)

best_idx, best_score = max(enumerate(chunks), key=lambda t: score_chunk(t[1]))
icl_document = chunks[best_idx]
print("Selected icl_document idx:", best_idx, "score:", best_score)
print(icl_document[:800])


Selected icl_document idx: 0 score: 1. Ficha técnica / Metadados | Nome | Feriados e Emendas de 2026 | |------------------|------------------------------------| | Área responsável | Departamento pessoal | | Categoria | Política de governança corporativa | 1.1. Revisões | Versão | Data | Alteração | |----------|------------|-------------| | 5 | 03/02/2026 | Testes... | | 1 | 02/02/2026 | Teste 01 | 2. Objetivo Divulgação do calendário de feriados nacionais e respectivos dias de emenda para o ano de 2026 . 3. Feriados Informamos que o escritório estará fechado nas seguintes datas, que correspondem a feriados nacionais e dias de emenda (em amarelo): | Periodo | Data | Dia da Semana | Feriado/Emenda | |-------------------|-----------------------|---------------------------|--------------------------------------------| | Ano Novo | 01/01/2026 02/01/2026 | Quinta-feira Sexta-feira | Feriado Nacional (Ano Novo) Emenda | | Carnaval | 16/02/2026 17/02/2026 | Segunda-feira Terca-feira | Emenda F

### 4.2 Construir um dataset seed (padrão SDG Hub)

O contrato do seed é o mesmo do notebook oficial: a coluna base é `document` (chunk) e adicionamos campos ICL.


In [7]:
import datasets
seed_data = datasets.Dataset.from_dict({"document": chunks})
print(seed_data)


/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['document'],
    num_rows: 1
})


### 4.3 Geração do `document_outline` e `icl_queries` usando **Blocks** do SDG Hub

Aqui está o ponto “100% SDG Hub”: usamos **blocks** (por exemplo, `LLMChatBlock`) para gerar campos e manter a execução/validação no padrão do framework.

> Observação: você pode trocar o provider/modelo via `flow.set_model_config()` ou variáveis no `.env`.


In [8]:
# Third Party
import pandas as pd

# First Party (SDG Hub)
from sdg_hub.core.blocks import LLMChatBlock
try:
    # Nem todas as versões expõem este bloco, então mantemos opcional.
    from sdg_hub.core.blocks import JSONStructureBlock
    HAS_JSON_BLOCK = True
except Exception:
    HAS_JSON_BLOCK = False

from sdg_hub import Flow

import textwrap


In [9]:
# Helper: configurar modelo nos blocks via Flow (mesmo padrão do knowledge_generation.ipynb)
def set_model_config(flow_object):
    model_provider = os.getenv("MODEL_PROVIDER", "hosted_vllm")
    print(f"Using model provider: {model_provider}")

    if model_provider == "hosted_vllm":
        vllm_model = os.getenv("VLLM_MODEL", "hosted_vllm/RedHatAI/gpt-oss-20b")
        vllm_api_base = os.getenv("VLLM_API_BASE", "http://granite-ai-inference-server-ocp.apps.cluster-jzfpx.jzfpx.sandbox2518.opentlc.com/v1")
        vllm_api_key = os.getenv("VLLM_API_KEY", "EMPTY")
        enable_reasoning = os.getenv("ENABLE_REASONING", "false").lower() in ("1","true","yes")
        flow_object.set_model_config(model=vllm_model, api_base=vllm_api_base, api_key=vllm_api_key, enable_reasoning=enable_reasoning)

    elif model_provider == "openai":
        openai_api_key = os.getenv("OPENAI_API_KEY")
        openai_model = os.getenv("OPENAI_MODEL", "openai/gpt-4")
        flow_object.set_model_config(model=openai_model, api_key=openai_api_key)

    elif model_provider == "openrouter":
        openai_api_key = os.getenv("OPENAI_API_KEY")
        openai_model = os.getenv("OPENAI_MODEL", "openai/gpt-4")
        flow_object.set_model_config(model=openai_model, api_key=openai_api_key, api_base="https://openrouter.ai/api/v1")

    elif model_provider == "ollama":
        ollama_model = os.getenv("OLLAMA_MODEL", "ollama/gemma2")
        ollama_api_base = os.getenv("OLLAMA_API_BASE", "http://localhost:11434")
        flow_object.set_model_config(model=ollama_model, api_base=ollama_api_base)

    elif model_provider == "maas":
        maas_model = os.getenv("MAAS_MODEL")
        maas_api_base = os.getenv("MAAS_API_BASE")
        maas_api_key = os.getenv("MAAS_API_KEY")
        flow_object.set_model_config(model=maas_model, api_base=maas_api_base, api_key=maas_api_key)

    return flow_object


#### 4.3.1 Dataset “ICL input row” (1 linha)

Vamos criar uma linha com o `icl_document` selecionado para pedir ao LLM:
- um outline curto
- um domínio (taxonomia simples)
- 3 perguntas (Q1 factual, Q2 procedural, Q3 edge case/comparação)

Depois distribuímos esses campos para todos os chunks (mesmo padrão do exemplo manual).


In [10]:
icl_df = pd.DataFrame([{
    "icl_document": icl_document
}])

icl_df.head()


,icl_document
0,1. Ficha técnica / Metadados | Nome | Feriados...


In [11]:
# SDG Hub LLMChatBlock espera uma coluna contendo uma LISTA de mensagens (formato OpenAI):
# [{"role":"user","content":"..."}]
#
# Então, criamos explicitamente duas colunas de mensagens:
# - messages_outline: para gerar OUTLINE/DOMAIN
# - messages_questions: para gerar JSON com icl_query_1..3

def build_messages_outline(icl_document: str):
    prompt = textwrap.dedent(f"""        You are generating seed metadata for knowledge tuning.
        Given the document excerpt below, produce:
        1) A concise 1-2 sentence document outline (high-level)
        2) A single domain label (choose ONE): Finance, Legal, IT, HR, Security, Operations, Healthcare, Education, Other

        Return exactly in this format:
        OUTLINE: <text>
        DOMAIN: <label>

        EXCERPT:
        {icl_document}
    """)
    return [{"role": "user", "content": prompt}]

def build_messages_questions(icl_document: str):
    prompt = textwrap.dedent(f"""        You are creating 3 high-quality, answerable questions for knowledge tuning.
        The questions MUST be answerable using ONLY the excerpt.

        Generate:
        - Q1: factual/definition
        - Q2: procedural/how-to or requirements
        - Q3: edge case / restriction / comparison

        Return STRICT JSON with keys: icl_query_1, icl_query_2, icl_query_3.

        EXCERPT:
        {icl_document}
    """)
    return [{"role": "user", "content": prompt}]

icl_df["messages_outline"] = icl_df["icl_document"].apply(build_messages_outline)
icl_df["messages_questions"] = icl_df["icl_document"].apply(build_messages_questions)

# Bloco 1: gerar document_outline + domain (texto)
outline_block = LLMChatBlock(
    block_name="gen_outline",
    input_cols=["messages_outline"],
    output_cols=["outline_response"],
)

# Bloco 2: gerar 3 perguntas (JSON para facilitar parsing)
questions_block = LLMChatBlock(
    block_name="gen_icl_questions",
    input_cols=["messages_questions"],
    output_cols=["questions_response"],
)


In [12]:
# Executar os blocks em modo "flow-like": instanciamos um Flow simples e executamos blocks sequencialmente.
# (Mantém o estilo SDG Hub: blocks com validação + execução padronizada.)
tmp_flow = Flow(metadata={"name":"Seed Semi-auto (study)","description":"Generate outline/domain + icl queries"})
tmp_flow.blocks = [outline_block, questions_block]
tmp_flow = set_model_config(tmp_flow)

# Rodar no dataframe (sdg_hub >=0.8 usa pandas internamente)
result_df = tmp_flow.generate(icl_df, max_concurrency=int(os.getenv("MAX_CONCURRENCY","4")))
result_df


Using model provider: hosted_vllm


[14:13:03] INFO     Auto-detected 2 LLM blocks for configuration: ['gen_icl_questions',         ]8;id=387023;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=515008;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py#242\242]8;;\
                    'gen_outline']                                                                                 

           INFO     Successfully configured 2 LLM blocks with: model:                           ]8;id=827757;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=10095;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py#300\300]8;;\
                    'hosted_vllm/RedHatAI/gpt-oss-20b', api_base:                                                  
                    'http://granite-ai-inference-server-ocp.apps.cluster-jzfpx.jzfpx.sandbox251                    
                    8.opentlc.com/v1', api_key: (redacted), enable_reasoning: 'False'                              

           INFO     Configured blocks: ['gen_icl_questions', 'gen_outline']                     ]8;id=615467;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=408154;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py#303\303]8;;\

[14:13:03] INFO     Using max_concurrency=50 for LLM requests                                      ]8;id=366684;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=12944;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py#419\419]8;;\

...

╭───────────────────────────────────────── gen_icl_questions - Complete ──────────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 1 → 1                                                                                                     │
│ Columns: 4 → 5                                                                                                  │
│ 🟢 Added: questions_response                                                                                    │
│ 📋 Final Columns: icl_document, messages_outline, messages_questions, outline_response, questions_response      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[14:13:08] INFO     Block 'gen_icl_questions' completed successfully: 1 samples, 5 columns         ]8;id=974727;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=215049;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py#266\266]8;;\

╭─────────────────────────────────────── Seed Semi-auto (study) - Complete ───────────────────────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ gen_outline          │ LLMChatBlock    │      1.69s │    1 → 1     │       +1        │     ✓      │           │
│ │ gen_icl_questions    │ LLMChatBlock    │      4.01s │    1 → 1     │       +1        │     ✓      │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 2 blocks        │      5.71s │   1 final    │     5 final     │    2/2     │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Flow 'Seed Semi-auto (study)' completed successfully: 1 final samples, 5 final ]8;id=781677;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=533488;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py#548\548]8;;\
                    columns                                                                                        

,icl_document,messages_outline,messages_questions,outline_response,questions_response
0,1. Ficha técnica / Metadados | Nome | Feriados...,"[{'role': 'user', 'content': 'You are generati...","[{'role': 'user', 'content': 'You are creating...",[{'content': 'OUTLINE: This document outlines ...,"[{'content': '{ ""icl_query_1"": ""What is the ..."


In [13]:
# LLMChatBlock retorna sempre LISTA de mensagens resposta.
# Cada item é um dict no formato OpenAI (role/content). Para n=1, usamos o primeiro.
outline_msg = result_df.loc[0, "outline_response"][0]
raw = outline_msg.get("content", "")

outline_match = re.search(r"OUTLINE:\s*(.+)", raw)
domain_match = re.search(r"DOMAIN:\s*(.+)", raw)

document_outline = (outline_match.group(1).strip() if outline_match else raw.strip())
domain = (domain_match.group(1).strip() if domain_match else "Other")

# Perguntas: conteúdo é JSON em string
questions_msg = result_df.loc[0, "questions_response"][0]
q_raw = questions_msg.get("content", "")

import json
try:
    q_obj = json.loads(q_raw)
except Exception:
    # fallback: tenta extrair linhas "icl_query_1:" etc.
    q_obj = {}
    for k in ["icl_query_1","icl_query_2","icl_query_3"]:
        m = re.search(rf"{k}\s*[:=]\s*(.+)", q_raw, flags=re.IGNORECASE)
        if m:
            q_obj[k] = m.group(1).strip()

icl_query_1 = q_obj.get("icl_query_1","").strip()
icl_query_2 = q_obj.get("icl_query_2","").strip()
icl_query_3 = q_obj.get("icl_query_3","").strip()

print("document_outline:", document_outline)
print("domain:", domain)
print("icl_query_1:", icl_query_1)
print("icl_query_2:", icl_query_2)
print("icl_query_3:", icl_query_3)


document_outline: This document outlines the national holidays and corresponding compensatory (emenda) days for 2026, detailing each date, weekday, and type of holiday, along with governance metadata and guidance on managing compensatory time.
domain: HR
icl_query_1: What is the stated objective of the "Feriados e Emendas de 2026" document?
icl_query_2: How are the emenda days for the office closure to be managed and compensated?
icl_query_3: Which holidays in 2026 include both a national holiday and an emenda day, and which holidays consist of only a single national holiday day?


### 4.4 Montar o seed final (map para todas as linhas) e salvar JSONL

Mesma lógica do exemplo oficial: todos os chunks recebem os mesmos campos ICL (para um estudo pequeno).


In [14]:
icl_fields = {
    "document_outline": document_outline,
    "icl_document": icl_document,
    "icl_query_1": icl_query_1,
    "icl_query_2": icl_query_2,
    "icl_query_3": icl_query_3,
    "domain": domain,
}

seed_data_final = seed_data.map(lambda _: icl_fields)
seed_path = os.getenv("SEED_DATA_PATH", "seed_data.jsonl")
seed_data_final.to_json(seed_path, orient="records", lines=True)
print("Saved:", seed_path)
print(seed_data_final[0])


Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 539.81ba/s]

Saved: seed_data.jsonl
{'document': '1. Ficha técnica / Metadados | Nome | Feriados e Emendas de 2026 | |------------------|------------------------------------| | Área responsável | Departamento pessoal | | Categoria | Política de governança corporativa | 1.1. Revisões | Versão | Data | Alteração | |----------|------------|-------------| | 5 | 03/02/2026 | Testes... | | 1 | 02/02/2026 | Teste 01 | 2. Objetivo Divulgação do calendário de feriados nacionais e respectivos dias de emenda para o ano de 2026 . 3. Feriados Informamos que o escritório estará fechado nas seguintes datas, que correspondem a feriados nacionais e dias de emenda (em amarelo): | Periodo | Data | Dia da Semana | Feriado/Emenda | |-------------------|-----------------------|---------------------------|--------------------------------------------| | Ano Novo | 01/01/2026 02/01/2026 | Quinta-feira Sexta-feira | Feriado Nacional (Ano Novo) Emenda | | Carnaval | 16/02/2026 17/02/2026 | Segunda-feira Terca-feira | Emenda 

## Step 5: Rodar um flow de geração do SDG Hub em cima do seed

Aqui seguimos o padrão do `knowledge_generation.ipynb`:
- `FlowRegistry.discover_flows()`
- escolher um flow (ex.: **Document Based Knowledge Tuning Dataset Generation Flow**)
- `Flow.from_yaml(flow_path)`
- `set_model_config(flow)`
- `flow.run(seed_df, ...)`
- salvar JSONL do output

> Para estudo, vamos:
> - subsample (ex.: 5 chunks)
> - ajustar runtime params se necessário


In [15]:
from sdg_hub import FlowRegistry
from datasets import load_dataset

FlowRegistry.discover_flows()
print("Total flows:", len(FlowRegistry.list_flows()))


[14:13:38] INFO     Discovered 11 flows                                                             ]8;id=255213;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/registry.py\registry.py]8;;\:]8;id=509731;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/registry.py#126\126]8;;\

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID                  ┃ Name                 ┃ Author               ┃ Tags                 ┃ Description          ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ clean-shadow-397    │ Advanced Japanese    │ SDG Hub Contributors │ question-generation, │ A comprehensive flow │
│                     │ Document Grounded    │                      │ knowledge-extractio… │ that generates       │
│                     │ Question-Answer      │                      │ qa-pairs,            │ high-quality         │
│                     │ Generation Flow for  │                      │ document-processing, │ question-answer      │
│                     │ Knowledge Tuning     │                      │ educational,         │ pairs from Japanese  │
│                     │                      │                      │ multilingual,        │ input documents      │
│                     │                      │                      │ japanese             │ using multiple LLM   │
│                     │                      │                      │                      │ blocks for question  │
│                     │                      │                      │                      │ generation, answer   │
│                     │                      │                      │                      │ synthesis, and       │
│                     │                      │                      │                      │ quality evaluation.  │
│ epic-jade-656       │ Extractive Summary   │ SDG Hub Contributors │ knowledge-tuning,    │ Generate extractive  │
│                     │ Knowledge Tuning     │                      │ document-internaliz… │ summary from the     │
│                     │ Dataset Generation   │                      │ question-generation, │ input document. Each │
│                     │ Flow                 │                      │ knowledge-extractiv… │ document is first    │
│                     │                      │                      │ qa-pairs,            │ converted into list  │
│                     │                      │                      │ extractive-summaries │ of knowledge         │
│                     │                      │                      │                      │ segments for         │
│                     │                      │                      │                      │ creating extractive  │
│                     │                      │                      │                      │ summary and then     │
│                     │                      │                      │                      │ annotated with       │
│                     │                      │                      │                      │ context,             │
│                     │                      │                      │                      │ relationship and     │
│                     │                      │                      │                      │ relevance. This is   │
│                     │                      │                      │                      │ then converted into  │
│                     │                      │                      │                      │ Question-Answer      │
│                     │                      │                      │                      │ pairs.               │
│ epic-jade-656-es    │ Extractive Summary   │ SDG Hub Contributors │ knowledge-tuning,    │ Generate extractive  │
│                     │ Knowledge Tuning     │                      │ document-internaliz… │ summary from the     │
│                     │ Dataset Generation   │                      │ question-generation, │ input document in    │
│                     │ Flow (Spanish)       │                      │ knowledge-extractiv… │ Spanish. Each        │
│                     │                      │          

Total flows: 11


In [16]:
# Carregar seed e subsample pequeno
seed_ds = load_dataset("json", data_files=seed_path, split="train")
subsample = int(os.getenv("SEED_DATA_SUBSAMPLE", "5"))
if subsample > 0:
    seed_ds = seed_ds.select(range(min(subsample, len(seed_ds))))

seed_df = seed_ds.to_pandas()
seed_df.head()


Generating train split: 1 examples [00:00, 383.25 examples/s]


,document,document_outline,icl_document,icl_query_1,icl_query_2,icl_query_3,domain
0,1. Ficha técnica / Metadados | Nome | Feriados...,This document outlines the national holidays a...,1. Ficha técnica / Metadados | Nome | Feriados...,"What is the stated objective of the ""Feriados ...",How are the emenda days for the office closure...,Which holidays in 2026 include both a national...,HR


In [17]:
# Escolher um flow pronto (document-based é simples e rápido)
flow_name = "Document Based Knowledge Tuning Dataset Generation Flow"

flow_path = FlowRegistry.get_flow_path(flow_name)
print("Flow path:", flow_path)

gen_flow = Flow.from_yaml(flow_path)
gen_flow = set_model_config(gen_flow)


Flow path: /opt/app-root/lib64/python3.12/site-packages/sdg_hub/flows/knowledge_infusion/enhanced_multi_summary_qa/doc_direct_qa/flow.yaml


[14:13:50] INFO     Loading flow from:                                                          ]8;id=746768;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/serialization.py\serialization.py]8;;\:]8;id=333854;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/serialization.py#60\60]8;;\
                    /opt/app-root/lib64/python3.12/site-packages/sdg_hub/flows/knowledge_infusi                    
                    on/enhanced_multi_summary_qa/doc_direct_qa/flow.yaml                                           

Using model provider: hosted_vllm


[14:13:50] INFO     Auto-detected 3 LLM blocks for configuration: ['answer_generation',         ]8;id=951053;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=105356;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py#242\242]8;;\
                    'eval_faithful_llm_chat', 'question_generation']                                               

           INFO     Successfully configured 3 LLM blocks with: model:                           ]8;id=18694;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=34681;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py#300\300]8;;\
                    'hosted_vllm/RedHatAI/gpt-oss-20b', api_base:                                                  
                    'http://granite-ai-inference-server-ocp.apps.cluster-jzfpx.jzfpx.sandbox251                    
                    8.opentlc.com/v1', api_key: (redacted), enable_reasoning: 'False'                              

           INFO     Configured blocks: ['answer_generation', 'eval_faithful_llm_chat',          ]8;id=374193;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=872842;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py#303\303]8;;\
                    'question_generation']                                                                         

In [19]:
df = seed_df.copy()

for b in gen_flow.blocks:
    df = b(df)  # executa bloco a bloco
    print(b.block_name, df.shape, list(df.columns))
    if b.block_name == "extract_questions":
        break

df.loc[0, "extract_questions_content"]

╭──────────────────────────────────────────── duplicate_document_col ─────────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: DuplicateColumnsBlock                                                                               │
│ Input Rows: 1                                                                                                   │
│ Input Columns: 7                                                                                                │
│ Column Names: document, document_outline, icl_document, icl_query_1, icl_query_2, icl_query_3, domain           │
│ Expected Output Columns: base_document                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── duplicate_document_col - Complete ───────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 1 → 1                                                                                                     │
│ Columns: 7 → 8                                                                                                  │
│ 🟢 Added: base_document                                                                                         │
│ 📋 Final Columns: base_document, document, document_outline, domain, icl_document, icl_query_1, icl_query_2,    │
│ icl_query_3                                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

duplicate_document_col (1, 8) ['document', 'document_outline', 'icl_document', 'icl_query_1', 'icl_query_2', 'icl_query_3', 'domain', 'base_document']


╭────────────────────────────────────────── question_generation_prompt ───────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: PromptBuilderBlock                                                                                  │
│ Input Rows: 1                                                                                                   │
│ Input Columns: 8                                                                                                │
│ Column Names: document, document_outline, icl_document, icl_query_1, icl_query_2, icl_query_3, domain,          │
│ base_document                                                                                                   │
│ Expected Output Columns: question_generation_prompt                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────── question_generation_prompt - Complete ─────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 1 → 1                                                                                                     │
│ Columns: 8 → 9                                                                                                  │
│ 🟢 Added: question_generation_prompt                                                                            │
│ 📋 Final Columns: base_document, document, document_outline, domain, icl_document, icl_query_1, icl_query_2,    │
│ icl_query_3, question_generation_prompt                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── extract_questions ───────────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: LLMResponseExtractorBlock                                                                           │
│ Input Rows: 1                                                                                                   │
│ Input Columns: 10                                                                                               │
│ Column Names: document, document_outline, icl_document, icl_query_1, icl_query_2, icl_query_3, domain,          │
│ base_document, question_generation_prompt, question_list                                                        │
│ Expected Output Columns: extract_questions_content                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[14:25:57] WARNING  Content field is None, using empty string instead           ]8;id=504709;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/blocks/llm/llm_response_extractor_block.py\llm_response_extractor_block.py]8;;\:]8;id=558896;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/blocks/llm/llm_response_extractor_block.py#163\163]8;;\

╭───────────────────────────────────────── extract_questions - Complete ──────────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 1 → 1                                                                                                     │
│ Columns: 10 → 11                                                                                                │
│ 🟢 Added: extract_questions_content                                                                             │
│ 📋 Final Columns: base_document, document, document_outline, domain, extract_questions_content, icl_document,   │
│ icl_query_1, icl_query_2, icl_query_3, question_generation_prompt, question_list                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

extract_questions (1, 11) ['document', 'document_outline', 'icl_document', 'icl_query_1', 'icl_query_2', 'icl_query_3', 'domain', 'base_document', 'question_generation_prompt', 'question_list', 'extract_questions_content']


''

In [20]:
# Rodar o flow com poucos docs
# runtime_params: você pode ajustar parâmetros específicos do flow (ex.: temperatura, max_tokens, etc.)
runtime_params = {
    "question_generation": {
        "temperature": 0.0,
        "max_tokens": 512,
        "n": 1,
    }
}

generated_df = gen_flow.generate(
    seed_df,
    runtime_params=runtime_params,
    max_concurrency=int(os.getenv("MAX_CONCURRENCY","4")),
)

generated_df.head()


[14:29:24] INFO     Using max_concurrency=50 for LLM requests                                      ]8;id=83885;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=543344;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py#419\419]8;;\

           INFO     Starting flow 'Document Based Knowledge Tuning Dataset Generation Flow' v2.0.0 ]8;id=711876;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=991571;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py#448\448]8;;\
                    with 1 samples across 14 blocks (max_concurrency=50)                                           

           INFO     Executing block 1/14: duplicate_document_col (DuplicateColumnsBlock)           ]8;id=309390;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=55869;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py#220\220]8;;\

╭──────────────────────────────────────────── duplicate_document_col ─────────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: DuplicateColumnsBlock                                                                               │
│ Input Rows: 1                                                                                                   │
│ Input Columns: 7                                                                                                │
│ Column Names: document, document_outline, icl_document, icl_query_1, icl_query_2, icl_query_3, domain           │
│ Expected Output Columns: base_document                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── duplicate_document_col - Complete ───────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 1 → 1                                                                                                     │
│ Columns: 7 → 8                                                                                                  │
│ 🟢 Added: base_document                                                                                         │
│ 📋 Final Columns: base_document, document, document_outline, domain, icl_document, icl_query_1, icl_query_2,    │
│ icl_query_3                                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

           INFO     Executing block 5/14: parse_question_list (TagParserBlock)                     ]8;id=81982;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=40440;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py#220\220]8;;\

╭────────────────────────────────────────────── parse_question_list ──────────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: TagParserBlock                                                                                      │
│ Input Rows: 1                                                                                                   │
│ Input Columns: 11                                                                                               │
│ Column Names: document, document_outline, icl_document, icl_query_1, icl_query_2, icl_query_3, domain,          │
│ base_document, question_generation_prompt, question_list, extract_questions_content                             │
│ Expected Output Columns: question                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── parse_question_list - Complete ─────────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 1 → 0                                                                                                     │
│ Columns: 11 → 0                                                                                                 │
│ 🔴 Removed: base_document, document, document_outline, domain, extract_questions_content, icl_document,         │
│ icl_query_1, icl_query_2, icl_query_3, question_generation_prompt, question_list                                │
│ 📋 Final Columns:                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────── Document Based Knowledge Tuning Dataset Generation Flow - Failed ────────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ duplicate_document_… │ DuplicateColum… │      0.00s │    1 → 1     │       +1        │     ✓      │           │
│ │ question_generation… │ PromptBuilderB… │      0.00s │    1 → 1     │       +1        │     ✓      │           │
│ │ question_generation  │ LLMChatBlock    │      3.97s │    1 → 1     │       +1        │     ✓      │           │
│ │ extract_questions    │ LLMResponseExt… │      0.01s │    1 → 1     │       +1        │     ✓      │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 4 blocks        │      3.98s │   0 final    │     0 final     │    4/4     │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

EmptyDatasetError: Block 'parse_question_list' received an empty dataset
Details: Dataset must contain at least one sample for processing

In [26]:
# 1) Ver o prompt que foi enviado ao modelo (é a referência do formato esperado)
print("\n=== question_generation_prompt (primeiros 2000 chars) ===\n")
print(df.loc[0, "question_generation_prompt"][:2000])

# 2) Ver o que o bloco de geração de perguntas produziu (antes do extract_questions)
print("\n=== question_list (raw) ===\n")
print(df.loc[0, "question_list"])

# 3) Ver o que o extract_questions produziu
print("\n=== extract_questions_content (raw) ===\n")
print(df.loc[0, "extract_questions_content"])

# 4) Ajuda: checar se está vazio mesmo (None, "", lista vazia etc.)
val = df.loc[0, "extract_questions_content"]
print("\n=== type/len ===")
print(type(val))
try:
    print("len:", len(val))
except Exception as e:
    print("len error:", e)


=== question_generation_prompt (primeiros 2000 chars) ===

[{'role': 'system', 'content': 'You are a very knowledgeable AI Assistant that will faithfully assist the user with their task.'}, {'role': 'user', 'content': 'Develop a series of educational questions from a chapter in a HR textbook.\n\nThe questions should:\n* Self-contained – understandable without needing to reference tables, figures, or specific text sections.\n* Focus on the provided example and follow the format and style of the provided examples.\n* Relevant to the subject – based on the textbook\'s domain (e.g., legal, scientific, etc.).\n* Independently answerable – avoid direct references to theorems, figures, or text numbers.\n* Varied in difficulty - Make difficult same as the provided examples.\n* Use same format as the provided examples.\n\nStrictly follow this format for each question your generate while responding\n\n[QUESTION]\n<Insert question here>\n[END]\n\nEach question and answer pair should stand alone 

In [ ]:
# Salvar resultado (pequeno dataset)
out_path = os.getenv("OUTPUT_DATA_PATH", "generated_knowledge.jsonl")
Path(out_path).parent.mkdir(parents=True, exist_ok=True)

generated_df.to_json(out_path, orient="records", lines=True)
print("Saved:", out_path)
print("Rows:", len(generated_df))


## Observações finais

- Se você quiser melhorar a robustez “semi-automática”, os próximos upgrades naturais são:
  1) gerar 2–3 candidatos a `icl_document` e pedir para o LLM escolher o melhor
  2) rodar um bloco “judge” rápido para checar se cada pergunta é respondível pelo trecho
  3) amostrar e revisar manualmente uma fração dos seeds

Mas, para estudos, este notebook já mantém:
- **contrato de seed data** compatível com o SDG Hub (`seed_data.jsonl`)
- **execução de flows via FlowRegistry** (padrão oficial)
- **uso de blocks SDG Hub** para automatizar outline/perguntas (sem fugir do framework)
